## Vision Transformer

In [ ]:
import os
import statistics

import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from datasets import load_dataset
from torch.utils.data import DataLoader
from torchvision.transforms.functional import to_pil_image
from tqdm import tqdm
from IPython.display import display
from PIL import Image

from models import ViT
from utils.hf_utils import ImageDatasetWithLabels

In [ ]:
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
def attention_rollout(attn_maps):
    result = None
    for attn in attn_maps:
        attn = attn.mean(dim=1)
        attn = attn.clamp(min=0)
        identity = torch.eye(
            attn.shape[-1],
            device=attn.device,
            dtype=attn.dtype,
        )
        attn = attn + identity
        attn = attn / (
            attn.sum(dim=-1, keepdim=True) + 1e-8
        )
        if result is None:
            result = attn
        else:
            result = attn @ result
    return result

In [ ]:
def load_checkpoint(checkpoint_path, device="cuda"):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model = ViT(
		img_size=checkpoint["img_size"],
		patch_size=checkpoint["patch_size"],
		num_classes=checkpoint["num_classes"],
		hidden_dim=checkpoint["hidden_dim"],
		num_blocks=checkpoint["num_blocks"],
		heads=checkpoint["heads"],
		head_dim=checkpoint["head_dim"],
		dropout=checkpoint["dropout"],
	)
    model = model.to(device)
    model.load_state_dict(checkpoint["vit"])
    return model

### Imagenet-1K

In [ ]:
imagenet_checkpoint_path = os.path.join("checkpoints", "imagenet-1k", "imagenet-1k_100.pth")

In [ ]:
imagenet_model = load_checkpoint(imagenet_checkpoint_path)

In [ ]:
hf_dataset = load_dataset("ILSVRC/imagenet-1k", split="validation")
transform = transforms.Compose([
    transforms.RandomResizedCrop((224, 224)),
	transforms.ToTensor(),
    transforms.Normalize(
		mean=(0.485, 0.456, 0.406),
		std=(0.229, 0.224, 0.225),
	),
])
imagenet_dataset = ImageDatasetWithLabels(hf_dataset, transform)

In [ ]:
imagenet_loader = DataLoader(
    imagenet_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

top1_accuracies = []
top5_accuracies = []

for _ in range(10):
    top1_correct = 0
    top5_correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in tqdm(
            imagenet_loader,
            desc="Evaluating",
            unit="batch",
        ):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = imagenet_model(images)
            top5_predictions = logits.topk(5, dim=1).indices

            top1_correct += (top5_predictions[:, 0] == labels).sum().item()
            top5_correct += (
                (top5_predictions == labels[:, None])
                .any(dim=1)
                .sum().item()
            )
            total += labels.size(0)

    top1_accuracy = 100 * top1_correct / total
    top5_accuracy = 100 * top5_correct / total

    top1_accuracies.append(top1_accuracy)
    top5_accuracies.append(top5_accuracy)

    print(f"Top-1: {top1_accuracy:.2f}%, Top-5: {top5_accuracy:.2f}%")

top1_mean = statistics.mean(top1_accuracies)
top1_std = statistics.stdev(top1_accuracies)

top5_mean = statistics.mean(top5_accuracies)
top5_std = statistics.stdev(top5_accuracies)

print(f"Top-1 Accuracy: {top1_mean:.2f}% ± {top1_std:.2f}%")
print(f"Top-5 Accuracy: {top5_mean:.2f}% ± {top5_std:.2f}%")

In [ ]:
random_idx = torch.randint(len(imagenet_dataset), (1,)).item()
imagenet_image, _ = imagenet_dataset[random_idx]
imagenet_image = imagenet_image.to(device)

imagenet_pil_image = to_pil_image(imagenet_image)
display(imagenet_pil_image)

In [ ]:
with torch.inference_mode():
	imagenet_model(imagenet_image.unsqueeze(0))

In [ ]:
attention_maps = [
    block.attention.attention_map
    for block in imagenet_model.encoder.blocks
]

rollout = attention_rollout(attention_maps)
cls_attention = rollout[:, 0, 1:]
grid_size = int(cls_attention.shape[-1] ** 0.5)
heatmap = cls_attention.reshape(
    -1, 1, grid_size, grid_size
)

heatmap = F.interpolate(
    heatmap,
    size=(224, 224),
    mode="bilinear",
    align_corners=False,
)

heatmap = (heatmap - heatmap.min()) / (
    heatmap.max() - heatmap.min() + 1e-8
)

imagenet_pil_image = to_pil_image(
    imagenet_image.squeeze(0).detach().cpu()
).convert("RGB")

heatmap = heatmap[0, 0].detach().cpu().numpy()

heatmap_colored = cm.jet(heatmap)[:, :, :3]
heatmap_colored = (heatmap_colored * 255).astype(np.uint8)

heatmap_pil = Image.fromarray(heatmap_colored)

overlay = Image.blend(
    imagenet_pil_image,
    heatmap_pil,
    alpha=0.4
)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(imagenet_pil_image)
axes[0].set_title("Image")

axes[1].imshow(heatmap_pil)
axes[1].set_title("Attention Rollout")

axes[2].imshow(overlay)
axes[2].set_title("Attention Visualization")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()